In [ ]:
%pip install gdown

In [ ]:
%pip install -r requirements.txt

In [ ]:
!apt-get update -qq && apt-get install -y -qq libxrender1 libxext6

In [ ]:
import gdown

file_id = '1C2gAjnGk9oQ6bbleUGSpZK3-HY5ifh6n'
url = f'https://drive.google.com/uc?id={file_id}'

gdown.download(url, output='processed2.zip', quiet=False)

In [ ]:
import zipfile
from pathlib import Path

zip_path = "processed2.zip"
out_dir = "./data/processed"

out_dir = Path(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Extracting {zip_path} -> {out_dir} ...")
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(out_dir)
print("Done.")

## Dataset

In [ ]:
import json
from functools import lru_cache
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset


class QMugsDataset(Dataset):
    def __init__(
        self,
        processed_path,
        split: str = "train",
        split_ratio: tuple = (0.8, 0.1, 0.1),
        seed: int = 42,
        cache_chunks: int = 2,
        max_samples: int = None,
    ):
        processed_path = Path(processed_path)
        self._processed_path = processed_path

        with open(processed_path / "meta.json") as f:
            meta = json.load(f)

        self._chunk_names = meta["chunk_files"]
        chunk_sizes = meta["chunk_sizes"]

        self._chunk_offsets = np.cumsum([0] + chunk_sizes[:-1])
        self._total = sum(chunk_sizes)

        rng = np.random.default_rng(seed)
        perm = rng.permutation(self._total)

        train_end = int(self._total * split_ratio[0])
        val_end = train_end + int(self._total * split_ratio[1])

        splits = {
            "train": perm[:train_end],
            "val":   perm[train_end:val_end],
            "test":  perm[val_end:],
        }
        if split not in splits:
            raise ValueError(f"split must be 'train', 'val', or 'test'; got '{split}'")
        split_indices = splits[split]

        # 청크 순서 셔플 + 청크 내 셔플 → DataLoader shuffle=False로 cache hit 보장
        chunk_of = np.searchsorted(self._chunk_offsets, split_indices, side="right") - 1
        chunk_order = rng.permutation(len(self._chunk_names))
        ordered = []
        for c in chunk_order:
            mask = chunk_of == c
            if mask.any():
                items = split_indices[mask].copy()
                rng.shuffle(items)
                ordered.append(items)
        self.indices = np.concatenate(ordered) if ordered else split_indices
        if max_samples is not None:
            self.indices = self.indices[:max_samples]

        with open(processed_path / "stats.json") as f:
            stats = json.load(f)
        prop_cols = list(stats.keys())
        self.mean = torch.tensor([stats[c]["mean"] for c in prop_cols], dtype=torch.float)
        self.std  = torch.tensor([stats[c]["std"]  for c in prop_cols], dtype=torch.float)

        self._load_chunk = lru_cache(maxsize=cache_chunks)(self._load_chunk_uncached)

        # 원자 수 사전계산 — size bucketing용 (init 1회, 이후 reshuffle에서 재사용)
        sizes_path = processed_path / "sizes.npy"
        if sizes_path.exists():
            all_sizes = np.load(sizes_path)
            self._sizes = all_sizes[self.indices]
        else:
            self._sizes = self._build_sizes()

        # reshuffle용 chunk 소속 캐시 — 매 에폭 searchsorted 재계산 방지
        self._chunk_of = np.searchsorted(self._chunk_offsets, self.indices, side="right") - 1

    def _load_chunk_uncached(self, chunk_idx: int) -> list:
        return torch.load(
            self._processed_path / self._chunk_names[chunk_idx],
            weights_only=False,
        )

    def _build_sizes(self) -> np.ndarray:
        chunk_of = np.searchsorted(self._chunk_offsets, self.indices, side="right") - 1
        sizes = np.empty(len(self.indices), dtype=np.int32)
        for c in np.unique(chunk_of):
            chunk_data = self._load_chunk(c)
            for pos in np.where(chunk_of == c)[0]:
                local = int(self.indices[pos]) - int(self._chunk_offsets[c])
                sizes[pos] = int((chunk_data[local].z != 1).sum())
        return sizes

    def reshuffle_indices(self):
        # 청크 순서는 랜덤, 청크 내부는 원자 수 오름차순 → size bucketing
        chunk_order = np.random.permutation(len(self._chunk_names))
        new_order = []
        for c in chunk_order:
            mask = self._chunk_of == c
            if mask.any():
                pos = np.where(mask)[0]
                sort = np.argsort(self._sizes[pos], kind="stable")
                new_order.append(pos[sort])
        if new_order:
            perm = np.concatenate(new_order)
            self.indices = self.indices[perm]
            self._sizes = self._sizes[perm]
            self._chunk_of = self._chunk_of[perm]

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int):
        global_idx = int(self.indices[idx])
        chunk_idx = int(np.searchsorted(self._chunk_offsets, global_idx, side="right") - 1)
        local_idx = global_idx - int(self._chunk_offsets[chunk_idx])
        data = self._load_chunk(chunk_idx)[local_idx].clone()
        data.p = (data.p_raw - self.mean) / self.std
        data.a_bin = (data.a[:10] > 0).float()

        # Heavy-atom only: remove H (atomic num 1)
        heavy_mask = data.z != 1
        if not heavy_mask.any():
            heavy_mask[0] = True
        heavy_idx = heavy_mask.nonzero(as_tuple=True)[0]

        old_to_new = torch.full((len(data.z),), -1, dtype=torch.long)
        old_to_new[heavy_idx] = torch.arange(len(heavy_idx), dtype=torch.long)

        ei = data.edge_index
        edge_mask = heavy_mask[ei[0]] & heavy_mask[ei[1]]
        data.z = data.z[heavy_idx]
        data.edge_index = old_to_new[ei[:, edge_mask]]
        data.bond_type = data.bond_type[edge_mask]
        data.num_nodes = len(heavy_idx)

        return data


## Precompute sizes.npy (dataset init 최적화)

In [ ]:
import numpy as np
import torch
import json
from pathlib import Path

processed_dir = "./data/processed"
sizes_path = Path(processed_dir) / "sizes.npy"

if sizes_path.exists():
    print("sizes.npy already exists, skipping.")
else:
    with open(f"{processed_dir}/meta.json") as f:
        meta = json.load(f)
    all_sizes = []
    for i, name in enumerate(meta["chunk_files"]):
        chunk = torch.load(f"{processed_dir}/{name}", weights_only=False)
        for data in chunk:
            all_sizes.append(int((data.z != 1).sum()))
        if (i + 1) % 10 == 0:
            print(f"{i + 1}/{len(meta['chunk_files'])} chunks done")
    np.save(sizes_path, np.array(all_sizes, dtype=np.int32))
    print(f"Done: {len(all_sizes)} molecules")


## Model

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import to_dense_adj, to_dense_batch

# 2026-07-27: absorbing([MASK]) noise -> marginal-transition noise로 교체.
# 근거: Vignac et al., "DiGress: Discrete Denoising diffusion for graph
# generation", ICLR 2023 (arXiv:2209.14734), Sec 4.1 / Theorem 4.1.
# t=T(완전 노이즈)에서도 그래프가 학습 데이터의 marginal 분포(대부분 no-bond)로
# 수렴하도록 설계 -> 모델이 "맥락 없을 때 기본값은 no-bond"를 노이즈 구조 자체에서
# 배움. MASK 토큰 방식은 이 사전 정보가 없어 맥락 부족 시 결합 과다예측 편향 발생
# (devlog.md 2026-07-26/27 진단 참조). K_X/K_E에 MASK 클래스 불필요 -> 원복.
#
# 2026-08-03: 방향족(aromatic) 클래스 제거. ground-truth 재구성 검증에서 방향족을
# 독립 클래스로 두는 표현이 이론적 validity 상한을 80.2%로 제한함을 실측 확인
# (전역 속성인 방향족성을 결합별 독립 예측으로는 일관되게 맞추기 어려움).
# preprocess.py에서 kekulize로 방향족 -> 명시적 단일/이중 교대 변환 후 재전처리.
# K_X: 0(padding), 1~9(C, O, N, S, P, F, Cl, Br, I) -> Total 10
# K_E: 0(no bond/padding), 1~3(Single, Double, Triple) -> Total 4
MAX_ATOMS = 50
K_X = 10
K_E = 4
T_STEPS = 150

ATOMIC_NUM_TO_CLS = {
    6: 1,  # C
    8: 2,  # O
    7: 3,  # N
    16: 4, # S
    15: 5, # P
    9: 6,  # F
    17: 7, # Cl
    35: 8, # Br
    53: 9  # I
}

NODE_DIM = 256
EDGE_DIM = 128
N_HEADS = 8
N_LAYERS = 6
FF_DIM = 512
T_DIM = 128


def _cosine_ac(T, s = 0.008):
    steps = torch.arange(T+1, dtype=torch.float64)
    f = torch.cos(((steps / T) + s) / (1 + s) * math.pi / 2) ** 2
    return (f / f[0]).float()[1:]


class MarginalNoiseSchedule(nn.Module):
    """Marginal-transition discrete noise schedule (DiGress Sec 4.1).

    Q^t = alpha^t I + beta^t * 1 m'  (m = 학습 데이터 marginal 클래스 분포)
    q(x^t | x^0=c) = ac_t * onehot(c) + (1-ac_t) * m
    Posterior q(x^{t-1}=i | x^t=j, x^0=c)는 Bayes rule로 직접 유도(closed form):
      propto [alpha_t*[i==j] + beta_t*m_j] * [ac_tm1*[i==c] + beta_tm1*m_i]
    (m_j는 관측값 j=x_t에서 계산한 스칼라. i에 대해 변하는 벡터가 아님에 주의 —
    이 부분 인덱싱 실수로 1차 로컬 검증에서 반증됐다가 수정 후 재검증됨.)
    이후 x0_probs(네트워크 예측)로 c에 대해 mixing (DiGress Eq.5).
    """

    def __init__(self, m_X, m_E, T = T_STEPS):
        super().__init__()
        ac = _cosine_ac(T)
        self.register_buffer("ac", ac)
        self.register_buffer("ac_prev", torch.cat([torch.ones(1), ac[:-1]]))
        self.register_buffer("m_X", m_X)  # [K_X], sums to 1, index 0(padding)=0
        self.register_buffer("m_E", m_E)  # [K_E], sums to 1
        self.T = T

    def q_sample(self, x0, t, m):
        shape = x0.shape
        B = t.shape[0]
        K = m.shape[-1]
        ac_t = self.ac[t-1].view(B, *([1] * (len(shape) - 1)))
        x0c = x0.clamp(min=0, max=K - 1)
        x0_onehot = F.one_hot(x0c, K).float()
        probs = ac_t.unsqueeze(-1) * x0_onehot + (1.0 - ac_t).unsqueeze(-1) * m
        probs = probs.clamp(min=0.0)
        probs = probs / probs.sum(-1, keepdim=True).clamp(min=1e-8)
        x_t = torch.multinomial(probs.reshape(-1, K), 1).view(shape)
        return x_t

    def posterior_sample(self, x_t, x0_probs, t, m):
        B = t.shape[0]
        shape = x_t.shape
        K = m.shape[-1]
        view = (B,) + (1,) * (len(shape) - 1)

        ac_t = self.ac[t-1].view(view)
        ac_tm1 = self.ac_prev[t-1].view(view)
        alpha_t = (ac_t / ac_tm1.clamp(min=1e-8)).clamp(max=1.0)
        beta_t = 1.0 - alpha_t

        # [Q^t]_{ij} = alpha_t*delta(i,j) + beta_t*m_j -- m_j는 관측값 j=x_t에서
        # 계산한 스칼라(i에 대해 상수), 벡터 m 전체가 아님.
        xt_clamped = x_t.clamp(min=0, max=K - 1)
        xt_onehot = F.one_hot(xt_clamped, K).float()
        m_xt = m[xt_clamped]
        term1 = (alpha_t.unsqueeze(-1) * xt_onehot + beta_t.unsqueeze(-1) * m_xt.unsqueeze(-1)).unsqueeze(-2)

        # [Q̄^{t-1}]_{ci} = ac_tm1*delta(c,i) + beta_tm1*m_i -- 여기는 i에 대해
        # 변하는 게 맞음 (i가 목적 상태이므로).
        eye = torch.eye(K, device=x_t.device, dtype=torch.float)
        beta_tm1 = 1.0 - ac_tm1
        term2 = ac_tm1.unsqueeze(-1).unsqueeze(-1) * eye + beta_tm1.unsqueeze(-1).unsqueeze(-1) * m.view(*([1]*len(view)), 1, K)

        unnorm = (term1 * term2).clamp(min=0.0)
        norm_c = unnorm.sum(-1, keepdim=True).clamp(min=1e-8)
        post_given_c = unnorm / norm_c

        result = (x0_probs.unsqueeze(-1) * post_given_c).sum(-2)
        result = result / result.sum(-1, keepdim=True).clamp(min=1e-8)

        stoch = torch.multinomial(result.reshape(-1, K), 1).view(shape)
        determ = x0_probs.argmax(dim=-1)
        t_mask = (t == 1).view(view).expand(shape)
        return torch.where(t_mask, determ, stoch)


class TimeEmbedding(nn.Module):

    def __init__(self, dim = T_DIM):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 2), nn.GELU(), nn.Linear(dim * 2, dim)
        )

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device).float() / (half - 1)
        )
        emb = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return self.mlp(emb)


class GTLayer(nn.Module):

    def __init__(self):
        super().__init__()
        assert NODE_DIM % N_HEADS == 0

        self.qkv = nn.Linear(NODE_DIM, NODE_DIM * 3, bias=False)
        self.e_bias = nn.Linear(EDGE_DIM, N_HEADS)
        self.attn_out = nn.Linear(NODE_DIM, NODE_DIM)
        self.norm1 = nn.LayerNorm(NODE_DIM)
        self.ff = nn.Sequential(
            nn.Linear(NODE_DIM, FF_DIM), nn.GELU(), nn.Linear(FF_DIM, NODE_DIM)
        )
        self.norm2 = nn.LayerNorm(NODE_DIM)
        # h_i+h_j만으로는 원자 쌍의 상호작용(어느 쌍이 결합하는지)을 구별하기 어려움
        # h_i*h_j를 추가해 쌍별 상호작용 신호를 명시적으로 제공 (대칭 유지)
        self.edge_ff = nn.Sequential(
            nn.Linear(NODE_DIM * 2 + EDGE_DIM, FF_DIM),
            nn.GELU(),
            nn.Linear(FF_DIM, EDGE_DIM)
        )
        self.enorm = nn.LayerNorm(EDGE_DIM)

    def forward(self, h, e, pad_mask):
        B, N, _ = h.shape
        nh = N_HEADS
        dh = NODE_DIM // N_HEADS

        Q, K, V = self.qkv(h).chunk(3, dim=-1)
        Q = Q.view(B, N, nh, dh).transpose(1, 2) # (B, nh, N, dh)
        K = K.view(B, N, nh, dh).transpose(1, 2)
        V = V.view(B, N, nh, dh).transpose(1, 2)

        scores = torch.einsum("bhid,bhjd->bhij", Q, K) / math.sqrt(dh)  # (B, nh, N, N)
        scores = scores + self.e_bias(e).permute(0, 3, 1, 2)
        scores = scores.masked_fill(pad_mask[:, None, None, :], float('-inf'))

        attn = scores.softmax(dim=-1)
        out  = torch.einsum("bhij,bhjd->bhid", attn, V)
        out = out.transpose(1, 2).reshape(B, N, -1)
        h = self.norm1(h + self.attn_out(out))
        h = self.norm2(h + self.ff(h))

        hi = h.unsqueeze(2).expand(-1, -1, N, -1) # (B, N, N, D)
        hj = h.unsqueeze(1).expand(-1, N, -1, -1) # (B, N, N, D)
        e = self.enorm(e + self.edge_ff(torch.cat([hi + hj, hi * hj, e], dim=-1)))

        h = h.masked_fill(pad_mask.unsqueeze(-1), 0.0)
        e_pad = pad_mask.unsqueeze(1) | pad_mask.unsqueeze(2)
        e = e.masked_fill(e_pad.unsqueeze(-1), 0.0)

        return h, e


class ScoreNetwork(nn.Module):

    def __init__(self):
        super().__init__()
        self.x_embed = nn.Embedding(K_X, NODE_DIM)
        self.e_embed = nn.Embedding(K_E, EDGE_DIM)
        self.t_embed = TimeEmbedding(T_DIM)
        self.t_proj  = nn.Linear(T_DIM, NODE_DIM)
        self.layers  = nn.ModuleList([GTLayer() for _ in range(N_LAYERS)])

        # 원자: padding(0) 제외한 K_X-1개 실제 원소 예측.
        # 결합: MASK 클래스가 없으므로 no-bond(0) 포함 전체 K_E개 예측.
        self.x_head  = nn.Sequential(nn.LayerNorm(NODE_DIM), nn.Linear(NODE_DIM, K_X - 1))
        self.e_head  = nn.Sequential(nn.LayerNorm(EDGE_DIM), nn.Linear(EDGE_DIM, K_E))

    def forward(self, X_t, E_t, node_mask, t):
        h   = self.x_embed(X_t) + self.t_proj(self.t_embed(t)).unsqueeze(1)
        e   = self.e_embed(E_t)
        pad = ~node_mask

        for layer in self.layers:
            h, e = layer(h, e, pad)

        x_logits = self.x_head(h)
        e_logits = self.e_head(e)
        e_logits = (e_logits + e_logits.transpose(1, 2)) / 2
        return x_logits, e_logits


class MoleculeGraphDiffusion(nn.Module):

    def __init__(self, m_X, m_E, T = T_STEPS, edge_loss_coeff = 0.2, class_weights = None, edge_class_weights = None):
        super().__init__()
        self.schedule = MarginalNoiseSchedule(m_X, m_E, T)
        self.net = ScoreNetwork()
        self.T = T
        self.edge_loss_coeff = edge_loss_coeff
        if class_weights is not None:
            self.register_buffer("class_weights", torch.tensor(class_weights, dtype=torch.float))
        else:
            self.register_buffer("class_weights", torch.ones(K_X - 1, dtype=torch.float))

        if edge_class_weights is not None:
            self.register_buffer("edge_class_weights", torch.tensor(edge_class_weights, dtype=torch.float))
        else:
            self.register_buffer("edge_class_weights", torch.ones(K_E, dtype=torch.float))

    def forward(self, batch):
        X0, E0, node_mask = self._to_dense(batch)
        B, N = X0.shape
        device = X0.device

        t = torch.randint(1, self.T + 1, (B,), device=device)

        # Marginal transition noise 주입
        X_t = self.schedule.q_sample(X0, t, self.schedule.m_X)
        E_t = self._sym(self.schedule.q_sample(E0, t, self.schedule.m_E))

        # 패딩 영역 복원 (패딩된 위치는 항상 0 유지)
        X_t = torch.where(node_mask, X_t, torch.zeros_like(X_t))
        e_mask = node_mask.unsqueeze(1) & node_mask.unsqueeze(2)
        E_t = torch.where(e_mask, E_t, torch.zeros_like(E_t))

        x_logits, e_logits = self.net(X_t, E_t, node_mask, t)

        # 원자: 실제 클래스 1..9 -> head 출력 0..8로 매핑
        nm = node_mask.view(-1)
        loss_x = F.cross_entropy(
            x_logits.view(-1, K_X - 1)[nm],
            (X0.view(-1)[nm] - 1).clamp(min=0),
            weight=self.class_weights
        )

        triu = torch.triu(torch.ones(N, N, device=device, dtype=torch.bool), diagonal=1)
        em = (node_mask.unsqueeze(2) & node_mask.unsqueeze(1)) & triu.unsqueeze(0)
        loss_e = F.cross_entropy(
            e_logits.view(-1, K_E)[em.view(-1)],
            E0.view(-1)[em.view(-1)],
            weight=self.edge_class_weights
        )

        loss = loss_x + self.edge_loss_coeff * loss_e
        return {
            "loss": loss,
            "loss_x": loss_x,
            "loss_e": loss_e
        }

    @torch.no_grad()
    def sample(self, n, n_atoms, device):
        node_mask = torch.ones(n, n_atoms, dtype=torch.bool, device=device)
        m_X, m_E = self.schedule.m_X, self.schedule.m_E

        # 초기 상태를 marginal 분포에서 샘플링 (DiGress Algorithm 2)
        X_t = torch.multinomial(m_X.expand(n * n_atoms, -1), 1).view(n, n_atoms).to(device)
        E_t = torch.multinomial(m_E.expand(n * n_atoms * n_atoms, -1), 1).view(n, n_atoms, n_atoms).to(device)

        E_t = self._sym(E_t)
        diag_mask = ~torch.eye(n_atoms, dtype=torch.bool, device=device).unsqueeze(0)
        E_t = torch.where(diag_mask, E_t, torch.zeros_like(E_t))

        for step in range(self.T, 0, -1):
            t = torch.full((n,), step, dtype=torch.long, device=device)
            xl, el = self.net(X_t, E_t, node_mask, t)

            # x_head는 실제 원자 클래스(1..9)만 예측 -> K_X 차원으로 복원 후 posterior 계산
            x0_probs_full = torch.zeros(n, n_atoms, K_X, device=device)
            x0_probs_full[..., 1:] = xl.softmax(-1)

            X_t = self.schedule.posterior_sample(X_t, x0_probs_full, t, m_X)
            E_t = self._sym(self.schedule.posterior_sample(E_t, el.softmax(-1), t, m_E))
            E_t = torch.where(diag_mask, E_t, torch.zeros_like(E_t))

        return X_t, E_t

    @staticmethod
    def _sym(E):
        N = E.shape[-1]
        triu = torch.triu(torch.ones(N, N, device=E.device, dtype=torch.bool), diagonal=1)
        E = E * triu
        return E + E.transpose(-1, -2)

    @staticmethod
    def _to_dense(batch):
        X0_raw, node_mask = to_dense_batch(batch.z, batch.batch)
        N = X0_raw.size(1)  # batch-local max atom count
        X0 = torch.zeros_like(X0_raw)
        for anum, cls in ATOMIC_NUM_TO_CLS.items():
            X0[X0_raw == anum] = cls

        ei = batch.edge_index
        uv = ei[0] < ei[1]
        bc = batch.bond_type[uv].argmax(dim=-1) + 1
        E0 = to_dense_adj(
            ei[:, uv], batch=batch.batch,
            edge_attr=bc.unsqueeze(-1).float(),
            max_num_nodes=N,
        ).squeeze(-1).long()
        E0 = E0 + E0.transpose(1, 2)

        return X0, E0, node_mask


## Train

In [ ]:
import copy
import io
import sys
import time
from pathlib import Path

import torch
from rdkit import Chem, RDLogger
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch_geometric.loader import DataLoader
from types import SimpleNamespace
from tqdm.notebook import tqdm



RDLogger.DisableLog("rdApp.*")

# 2026-08-03: 방향족 클래스 제거(kekulize로 명시적 단일/이중 교대 변환).
_BOND_TYPE = {
    1: Chem.rdchem.BondType.SINGLE,
    2: Chem.rdchem.BondType.DOUBLE,
    3: Chem.rdchem.BondType.TRIPLE,
}
_CLS_TO_ANUM = {v: k for k, v in ATOMIC_NUM_TO_CLS.items()}

# [R-1] 원자가 위반 판정용 원소별 표준 원자가 (RDKit 실측: C4/N3/O2/S2/P3/할로겐1).
# index 0(padding)=0.
_PT = Chem.GetPeriodicTable()
_MAX_VALENCE = torch.zeros(K_X)
for _anum, _cls in ATOMIC_NUM_TO_CLS.items():
    _MAX_VALENCE[_cls] = _PT.GetDefaultValence(_anum)


class _Tee(io.TextIOBase):

    def __init__(self, stream, file_path):
        self._stream = stream
        self._f = open(file_path, "a", encoding="utf-8")

    def write(self, text):
        self._stream.write(text)
        self._f.write(text)
        return len(text)

    def flush(self):
        self._stream.flush()
        self._f.flush()

    def close(self):
        self._f.close()


class EMA:

    def __init__(self, model, decay = 0.9999):
        self.model = copy.deepcopy(model).eval()
        self.decay = decay

    def update(self, model):
        with torch.no_grad():
            for ema_p, p in zip(self.model.parameters(), model.parameters()):
                ema_p.mul_(self.decay).add_(p.data, alpha=1.0 - self.decay)

    def state_dict(self):
        return self.model.state_dict()

    def load_state_dict(self, sd):
        self.model.load_state_dict(sd)


def tensors_to_mol(X, E):
    mol = Chem.RWMol()
    # MASK 클래스 없음(marginal transition) -> 패딩(0)만 제외, 나머지 1..K_X-1 전부 실제 원자
    real = ((X > 0) & (X < K_X)).nonzero(as_tuple=True)[0].tolist()
    if len(real) < 2:
        return None

    idx_map = {}
    for ai in real:
        anum = _CLS_TO_ANUM.get(int(X[ai]))
        if anum is None:
            return None
        idx_map[ai] = mol.AddAtom(Chem.Atom(anum))

    for i, ai in enumerate(real):
        for aj in real[i + 1:]:
            bc = int(E[ai, aj])
            # 0(padding/no bond)만 제외 -- MASK 클래스 없음
            if bc == 0:
                continue
            btype = _BOND_TYPE.get(bc)
            if btype is None:
                continue
            mol.AddBond(idx_map[ai], idx_map[aj], btype)

    try:
        Chem.SanitizeMol(mol)

        # 꼼수 방지: 고립 원자들의 단순 나열로 100% validity를 교란하는 모델 방어 필터
        frags = Chem.GetMolFrags(mol, asMols=True)
        if len(frags) > 1:
            max_frag_size = max(len(f.GetAtoms()) for f in frags)
            if max_frag_size / len(mol.GetAtoms()) < 0.85:
                return None

        return mol.GetMol()
    except Exception:
        return None


def atom_valence_violations(X, E):
    """[R-1] Tensor 기반 원자가 위반 계산 -- RDKit SanitizeMol에 의존하지 않음
    (sanitize는 첫 실패에서 예외를 던져 위반 개수를 셀 수 없음).
    E의 클래스값이 곧 결합 차수(1=단일,2=이중,3=삼중)이므로 행 합이 그 원자가
    실제로 쓰고 있는 총 원자가다.

    반환: (위반 원자 수, 전체(패딩 제외) 원자 수)
    """
    real = X > 0
    n_atoms = int(real.sum().item())
    if n_atoms == 0:
        return 0, 0
    degree = E.float().sum(dim=-1)
    max_val = _MAX_VALENCE[X.clamp(min=0, max=K_X - 1)]
    violations = int(((degree > max_val) & real).sum().item())
    return violations, n_atoms


def analyze_molecule(X, E):
    """[R-1] 분자 하나에 대해 strict/largest-fragment validity와 fragment 비율을 계산.

    - strict: 현행 기준 그대로 유지 (SanitizeMol 통과 + 최대 조각 비율 >= 0.85,
      고립 원자 나열로 validity를 교란하는 꼼수 방지 필터 포함).
    - largest-fragment: 최대 조각만 추출해 그것만 sanitize (MOSES/DiGress 등
      문헌이 쓰는 관대한 정의, 논문 대비 비교용 -- [R-9]).
    """
    real = ((X > 0) & (X < K_X)).nonzero(as_tuple=True)[0].tolist()
    if len(real) < 2:
        return {"strict_valid": False, "frag_valid": False, "frag_ratio": 0.0, "mol": None}

    mol = Chem.RWMol()
    idx_map = {}
    for ai in real:
        anum = _CLS_TO_ANUM.get(int(X[ai]))
        if anum is None:
            return {"strict_valid": False, "frag_valid": False, "frag_ratio": 0.0, "mol": None}
        idx_map[ai] = mol.AddAtom(Chem.Atom(anum))

    for i, ai in enumerate(real):
        for aj in real[i + 1:]:
            bc = int(E[ai, aj])
            if bc == 0:
                continue
            btype = _BOND_TYPE.get(bc)
            if btype is None:
                continue
            mol.AddBond(idx_map[ai], idx_map[aj], btype)

    raw_mol = mol.GetMol()

    try:
        frags = Chem.GetMolFrags(raw_mol, asMols=True, sanitizeFrags=False)
    except Exception:
        frags = ()
    if not frags:
        return {"strict_valid": False, "frag_valid": False, "frag_ratio": 0.0, "mol": None}

    largest = max(frags, key=lambda f: f.GetNumAtoms())
    frag_ratio = largest.GetNumAtoms() / len(real)

    strict_valid = False
    strict_mol = None
    try:
        cand = Chem.Mol(raw_mol)
        Chem.SanitizeMol(cand)
        if frag_ratio >= 0.85:
            strict_valid = True
            strict_mol = cand
    except Exception:
        pass

    frag_valid = False
    frag_mol = None
    try:
        cand = Chem.Mol(largest)
        Chem.SanitizeMol(cand)
        frag_valid = True
        frag_mol = cand
    except Exception:
        pass

    return {
        "strict_valid": strict_valid,
        "frag_valid": frag_valid,
        "frag_ratio": frag_ratio,
        "mol": strict_mol if strict_mol is not None else frag_mol,
    }


def build_size_distribution(dataset, n_sample = 2000):
    sizes = []
    ci = 0
    while len(sizes) < n_sample and ci < len(dataset._chunk_names):
        for data in dataset._load_chunk(ci):
            n = int((data.z != 1).sum())
            if 0 < n <= MAX_ATOMS:
                sizes.append(n)
            if len(sizes) >= n_sample:
                break
        ci += 1
    return sizes


def run_epoch(model, loader, device, train, epoch, optimizer=None, ema=None):
    model.train(train)
    totals = {
        "loss": 0.0,
        "loss_x": 0.0,
        "loss_e": 0.0
    }

    with torch.set_grad_enabled(train):
        for batch in tqdm(loader, desc=f"Epoch {epoch}", leave=False):
            batch = batch.to(device)
            result = model(batch)

            if train:
                optimizer.zero_grad()
                result["loss"].backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1.0)
                optimizer.step()
                if ema is not None:
                    ema.update(model)

            for k in totals:
                totals[k] += result[k].item()

    n = len(loader)
    return {k: v / n for k, v in totals.items()}


@torch.no_grad()
def evaluate_validity(model, size_dist, n_eval, device):
    """[R-1] validity는 median 29원자에서 v^29로 수렴하는 곱셈 지표라 이 성능
    구간에서 해상도가 사실상 0(0.80과 0.95의 validity가 둘 다 0.000으로 찍힘).
    validity(strict/frag)에 더해 원자 단위 연속 지표(위반율)를 주 지표로 함께
    반환한다 -- 관측 개수가 n_eval(분자) -> 원자 수(약 30배)로 늘어나 훨씬
    이른 시점에 개선 추세를 볼 수 있다.
    """
    import random
    from collections import Counter

    model.eval()
    sizes = random.choices(size_dist, k=n_eval)
    size_counts = Counter(sizes)

    n_strict = 0
    n_frag = 0
    strict_mols = []
    total_atoms = 0
    total_violations = 0
    violations_per_mol = []
    frag_ratios = []

    for n_atoms, count in size_counts.items():
        if n_atoms > MAX_ATOMS:
            continue
        X, E = model.sample(count, n_atoms, device)
        X_cpu, E_cpu = X.cpu(), E.cpu()
        del X, E
        torch.cuda.empty_cache()
        for i in range(count):
            xi, ei = X_cpu[i], E_cpu[i]

            n_v, n_a = atom_valence_violations(xi, ei)
            total_violations += n_v
            total_atoms += n_a
            violations_per_mol.append(n_v)

            info = analyze_molecule(xi, ei)
            frag_ratios.append(info["frag_ratio"])
            if info["strict_valid"]:
                n_strict += 1
                strict_mols.append(info["mol"])
            if info["frag_valid"]:
                n_frag += 1

    if strict_mols:
        smiles = {Chem.MolToSmiles(m) for m in strict_mols}
        uniqueness = len(smiles) / len(strict_mols)
    else:
        uniqueness = 0.0

    return {
        "validity_strict": n_strict / n_eval,
        "validity_frag": n_frag / n_eval,
        "uniqueness": uniqueness,
        "atom_violation_rate": (total_violations / total_atoms) if total_atoms > 0 else float("nan"),
        "mean_violations_per_mol": (sum(violations_per_mol) / len(violations_per_mol)) if violations_per_mol else float("nan"),
        "mean_frag_ratio": (sum(frag_ratios) / len(frag_ratios)) if frag_ratios else float("nan"),
    }


@torch.no_grad()
def diagnostic_fixed_t(model, val_loader, device, t_values=(10, 75, 140), n_batches=4):
    """[R-2] 고정 t에서 q_sample -> forward 한 번만으로 x0 재구성 정확도를
    측정한다 (150-step 전체 샘플링 불필요, forward pass 몇 번이라 비용 거의 0).
    원자/결합/무결합 정확도를 분리해서, "맥락이 거의 없을 때(t 큼) 결합 쪽으로
    쏠리는 편향"이 marginal transition에서도 남아있는지 확인한다
    (devlog 2026-07-26 absorbing 진단 방법론을 marginal에 재적용).
    """
    model.eval()
    stats = {t: {"atom_c": 0, "atom_n": 0, "bond_c": 0, "bond_n": 0, "nobond_c": 0, "nobond_n": 0}
              for t in t_values}

    it = iter(val_loader)
    for _ in range(n_batches):
        try:
            batch = next(it)
        except StopIteration:
            break
        batch = batch.to(device)
        X0, E0, node_mask = model._to_dense(batch)
        B, N = X0.shape
        e_mask = node_mask.unsqueeze(1) & node_mask.unsqueeze(2)
        triu = torch.triu(torch.ones(N, N, device=device, dtype=torch.bool), diagonal=1)
        em = e_mask & triu.unsqueeze(0)

        for t_val in t_values:
            t = torch.full((B,), t_val, device=device, dtype=torch.long)
            X_t = model.schedule.q_sample(X0, t, model.schedule.m_X)
            E_t = model._sym(model.schedule.q_sample(E0, t, model.schedule.m_E))
            X_t = torch.where(node_mask, X_t, torch.zeros_like(X_t))
            E_t = torch.where(e_mask, E_t, torch.zeros_like(E_t))

            x_logits, e_logits = model.net(X_t, E_t, node_mask, t)
            x_pred = x_logits.argmax(-1) + 1
            e_pred = e_logits.argmax(-1)

            s = stats[t_val]
            s["atom_c"] += int(((x_pred == X0) & node_mask).sum().item())
            s["atom_n"] += int(node_mask.sum().item())

            bond_mask = em & (E0 > 0)
            nobond_mask = em & (E0 == 0)
            s["bond_c"] += int(((e_pred == E0) & bond_mask).sum().item())
            s["bond_n"] += int(bond_mask.sum().item())
            s["nobond_c"] += int(((e_pred == E0) & nobond_mask).sum().item())
            s["nobond_n"] += int(nobond_mask.sum().item())

    result = {}
    for t_val, s in stats.items():
        result[t_val] = {
            "atom_acc":   s["atom_c"] / s["atom_n"] if s["atom_n"] > 0 else float("nan"),
            "bond_acc":   s["bond_c"] / s["bond_n"] if s["bond_n"] > 0 else float("nan"),
            "nobond_acc": s["nobond_c"] / s["nobond_n"] if s["nobond_n"] > 0 else float("nan"),
        }
    return result


def _print_eval_block(tag, metrics, diag):
    print(
        f"    [{tag}] valid(strict)={metrics['validity_strict']:.3f} "
        f"valid(frag)={metrics['validity_frag']:.3f} "
        f"atomViolRate={metrics['atom_violation_rate']:.3f} "
        f"meanViol/mol={metrics['mean_violations_per_mol']:.2f} "
        f"fragRatio={metrics['mean_frag_ratio']:.3f} "
        f"uniq={metrics['uniqueness']:.3f}"
    )
    parts = " | ".join(
        f"t={t_val:<3} atom={d['atom_acc']:.3f} bond={d['bond_acc']:.3f} nobond={d['nobond_acc']:.3f}"
        for t_val, d in diag.items()
    )
    print(f"    [{tag}] {parts}")


def main():
    # 2026-08-25: docs/review.md(Opus 리뷰) R-1/R-2/R-3 반영.
    # - R-1: validity는 median 29원자에서 v^29로 수렴해 이 성능 구간에서
    #   해상도가 사실상 0 -- 원자 단위 연속 지표(atomViolRate 등)를 주 지표로 추가.
    # - R-2: 고정 t={10,75,140}에서 forward pass만으로 원자/결합/무결합 정확도 로깅.
    # - R-3: ema_decay를 step 예산(150000/32*5≈23,437)에 안전한 0.999로 낮추고,
    #   raw model 지표를 EMA와 나란히 로깅.
    args = SimpleNamespace(
        processed_dir = "./data/processed",
        save_dir      = "./checkpoints_diffusion_marginal_full_smoke",
        epochs        = 5,
        batch_size    = 32,
        lr            = 1e-4,
        ema_decay     = 0.999,
        save_every    = 1,
        eval_every    = 1,
        n_eval        = 128,
        num_workers   = 0,
        max_samples   = 150000,
        log_file      = "./train_diffusion_marginal_full_smoke.log",
        resume        = None
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    save_dir = Path(args.save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    tee = _Tee(sys.stdout, args.log_file)
    sys.stdout = tee

    print(f"Device: {device}")
    print(f"Log: {args.log_file}")

    print("Loading datasets ...")
    train_set = QMugsDataset(
        args.processed_dir, split="train",
        max_samples=args.max_samples
    )
    val_set = QMugsDataset(
        args.processed_dir, split="val",
        max_samples=args.max_samples // 5 if args.max_samples else None
    )
    train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=False,
                              num_workers=args.num_workers)
    val_loader   = DataLoader(val_set,   batch_size=args.batch_size, shuffle=False,
                              num_workers=args.num_workers)
    print(f"  Train: {len(train_set):,}  Val: {len(val_set):,}")

    print("Building size distribution ...")
    size_dist = build_size_distribution(train_set)
    print(f"  Size range: {min(size_dist)} ~ {max(size_dist)} atoms  "
          f"(median {sorted(size_dist)[len(size_dist)//2]})")

    # marginal transition noise용 클래스 분포 계산 (DiGress Sec 4.1).
    # inverse-frequency loss reweighting은 2026-07-26 ablation에서 효과 없음이
    # 확인됐고 논문에도 없는 기법이라 제거 -- 희소성 보정은 노이즈 자체(marginal
    # 분포로 수렴)가 담당하도록 함.
    print("Computing marginal class distributions (node/edge) ...")
    x_counts = torch.zeros(K_X, dtype=torch.float)
    n_weight_samples = min(5000, len(train_set))
    for i in range(n_weight_samples):
        data = train_set[i]
        for anum in data.z:
            cls_idx = ATOMIC_NUM_TO_CLS.get(int(anum), 0)
            x_counts[cls_idx] += 1
    x_counts[0] = 0.0  # padding은 marginal에 포함하지 않음
    m_X = x_counts / x_counts.sum().clamp(min=1.0)
    print(f"  m_X: {m_X.tolist()}")

    e_counts = torch.zeros(K_E, dtype=torch.float)
    n_edge_samples = min(1000, len(train_set))
    for i in range(n_edge_samples):
        data = train_set[i]
        btype = data.bond_type.argmax(dim=-1) + 1
        for bt in btype:
            e_counts[int(bt)] += 1
        n_atoms = len(data.z)
        total_possible = n_atoms * (n_atoms - 1)
        no_bond = total_possible - len(btype)
        e_counts[0] += no_bond
    m_E = e_counts / e_counts.sum().clamp(min=1.0)
    print(f"  m_E: {m_E.tolist()}")

    model = MoleculeGraphDiffusion(m_X, m_E).to(device)
    ema = EMA(model, decay=args.ema_decay)
    ema.model.to(device)

    optimizer = AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=1e-6)

    best_val_loss = float('inf')
    start_epoch = 1

    if args.resume:
        ckpt = torch.load(args.resume, map_location=device)
        model.load_state_dict(ckpt["model"])
        ema.load_state_dict(ckpt["ema"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        start_epoch = ckpt["epoch"] + 1
        best_val_loss = ckpt.get("best_val_loss", float('inf'))
        print(f"  Resumed from epoch {ckpt['epoch']}  (best val loss: {best_val_loss:.4f})")

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Parameters: {n_params:,}")

    print(f"\n{'Epoch':>6}  {'T-loss':>8} {'T-x':>8} {'T-e':>8}  "
          f"{'V-loss':>8} {'V-x':>8} {'V-e':>8}  Time")
    print("-" * 90)

    for epoch in range(start_epoch, args.epochs + 1):
        t0 = time.time()
        train_set.reshuffle_indices()

        train = run_epoch(model, train_loader, device, train=True, epoch=epoch, optimizer=optimizer, ema=ema)
        val = run_epoch(ema.model, val_loader, device, train=False, epoch=epoch)

        scheduler.step()

        if val["loss"] < best_val_loss:
            best_val_loss = val["loss"]
            torch.save(ema.model.state_dict(), save_dir / "best.pt")

        if args.save_every > 0 and epoch % args.save_every == 0:
            torch.save({
                "epoch": epoch,
                "model": model.state_dict(),
                "ema": ema.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "best_val_loss": best_val_loss
            }, save_dir / f"ckpt_epoch{epoch:04d}.pt")

        elapsed = time.time() - t0
        print(
            f"{epoch:>6}  "
            f"{train['loss']:>8.4f} {train['loss_x']:>8.4f} {train['loss_e']:>8.4f}  "
            f"{val['loss']:>8.4f} {val['loss_x']:>8.4f} {val['loss_e']:>8.4f}  "
            f"{elapsed:>6.1f}s"
        )

        # [R-3] raw model 지표를 EMA와 나란히 로깅 -- 2026-07-26에 이 비교가
        # EMA 오염 가설을 기각하는 데 결정적이었던 전례를 재사용.
        if args.eval_every > 0 and epoch % args.eval_every == 0:
            ema_metrics = evaluate_validity(ema.model, size_dist, args.n_eval, device)
            ema_diag = diagnostic_fixed_t(ema.model, val_loader, device)
            _print_eval_block("EMA", ema_metrics, ema_diag)

            raw_metrics = evaluate_validity(model, size_dist, args.n_eval, device)
            raw_diag = diagnostic_fixed_t(model, val_loader, device)
            _print_eval_block("RAW", raw_metrics, raw_diag)

    print(f"\nDone. Best val loss: {best_val_loss:.4f}")
    sys.stdout = tee._stream
    tee.close()


main()


## Test / Inference

In [ ]:
import random
from collections import Counter
from IPython.display import display
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw

RDLogger.DisableLog("rdApp.*")

CKPT_PATH = "./checkpoints_diffusion_marginal_full_smoke/best.pt"
N_EVAL = 64
PROCESSED_DIR = "./data/processed"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# m_X/m_E는 학습된 체크포인트의 buffer에 저장되어 있으므로, 인스턴스화 시에는
# 올바른 shape의 placeholder만 넣으면 load_state_dict가 실제 값으로 덮어씀.
_m_X_placeholder = torch.ones(K_X) / K_X
_m_E_placeholder = torch.ones(K_E) / K_E

ema_model = MoleculeGraphDiffusion(_m_X_placeholder, _m_E_placeholder).to(device)
ckpt = torch.load(CKPT_PATH, map_location=device)
if isinstance(ckpt, dict) and "ema" in ckpt:
    ema_model.load_state_dict(ckpt["ema"])
    print(f"Loaded EMA weights from checkpoint (epoch {ckpt.get('epoch', '?')})")
elif isinstance(ckpt, dict) and "model" in ckpt:
    ema_model.load_state_dict(ckpt["model"])
    print(f"Loaded model weights from checkpoint (epoch {ckpt.get('epoch', '?')})")
else:
    ema_model.load_state_dict(ckpt)
    print("Loaded state dict from best.pt")
ema_model.eval()

# 2026-08-03: 방향족 클래스 제거(kekulize로 명시적 단일/이중 교대 변환).
_BOND_TYPE = {
    1: Chem.rdchem.BondType.SINGLE,
    2: Chem.rdchem.BondType.DOUBLE,
    3: Chem.rdchem.BondType.TRIPLE,
}
_CLS_TO_ANUM = {v: k for k, v in ATOMIC_NUM_TO_CLS.items()}

# [R-1] 원자가 위반 판정용 원소별 표준 원자가 (RDKit 실측: C4/N3/O2/S2/P3/할로겐1).
# CPU 텐서로 유지 -- 아래서 X/E를 .cpu()로 옮긴 뒤 인덱싱한다.
_PT = Chem.GetPeriodicTable()
_MAX_VALENCE = torch.zeros(K_X)
for _anum, _cls in ATOMIC_NUM_TO_CLS.items():
    _MAX_VALENCE[_cls] = _PT.GetDefaultValence(_anum)


def atom_valence_violations(X, E):
    real = X > 0
    n_atoms = int(real.sum().item())
    if n_atoms == 0:
        return 0, 0
    degree = E.float().sum(dim=-1)
    max_val = _MAX_VALENCE[X.clamp(min=0, max=K_X - 1)]
    violations = int(((degree > max_val) & real).sum().item())
    return violations, n_atoms


def analyze_molecule(X, E):
    """strict(현행, 꼼수 방지 포함) / largest-fragment(문헌 비교용) validity를
    함께 계산. train3.py의 동일 함수와 로직 동기화됨."""
    real = ((X > 0) & (X < K_X)).nonzero(as_tuple=True)[0].tolist()
    if len(real) < 2:
        return {"strict_valid": False, "frag_valid": False, "frag_ratio": 0.0, "mol": None}

    mol = Chem.RWMol()
    idx_map = {}
    for ai in real:
        anum = _CLS_TO_ANUM.get(int(X[ai]))
        if anum is None:
            return {"strict_valid": False, "frag_valid": False, "frag_ratio": 0.0, "mol": None}
        idx_map[ai] = mol.AddAtom(Chem.Atom(anum))
    for i, ai in enumerate(real):
        for aj in real[i + 1:]:
            bc = int(E[ai, aj])
            if bc == 0:
                continue
            btype = _BOND_TYPE.get(bc)
            if btype is None:
                continue
            mol.AddBond(idx_map[ai], idx_map[aj], btype)
    raw_mol = mol.GetMol()

    try:
        frags = Chem.GetMolFrags(raw_mol, asMols=True, sanitizeFrags=False)
    except Exception:
        frags = ()
    if not frags:
        return {"strict_valid": False, "frag_valid": False, "frag_ratio": 0.0, "mol": None}

    largest = max(frags, key=lambda f: f.GetNumAtoms())
    frag_ratio = largest.GetNumAtoms() / len(real)

    strict_valid = False
    strict_mol = None
    try:
        cand = Chem.Mol(raw_mol)
        Chem.SanitizeMol(cand)
        if frag_ratio >= 0.85:
            strict_valid = True
            strict_mol = cand
    except Exception:
        pass

    frag_valid = False
    frag_mol = None
    try:
        cand = Chem.Mol(largest)
        Chem.SanitizeMol(cand)
        frag_valid = True
        frag_mol = cand
    except Exception:
        pass

    return {
        "strict_valid": strict_valid,
        "frag_valid": frag_valid,
        "frag_ratio": frag_ratio,
        "mol": strict_mol if strict_mol is not None else frag_mol,
    }


def _build_size_dist(processed_dir, n_sample=500, max_samples=3000):
    dataset = QMugsDataset(processed_dir, split="train", max_samples=max_samples)
    sizes = []
    ci = 0
    while len(sizes) < n_sample and ci < len(dataset._chunk_names):
        for data in dataset._load_chunk(ci):
            n = int((data.z != 1).sum())
            if 0 < n <= MAX_ATOMS:
                sizes.append(n)
            if len(sizes) >= n_sample:
                break
        ci += 1
    return sizes

size_dist = _build_size_dist(PROCESSED_DIR)
sizes = random.choices(size_dist, k=N_EVAL)
size_counts = Counter(sizes)

n_strict = 0
n_frag = 0
strict_mols = []
total_atoms = 0
total_violations = 0
with torch.no_grad():
    for n_atoms, count in size_counts.items():
        if n_atoms > MAX_ATOMS:
            continue
        X, E = ema_model.sample(count, n_atoms, device)
        X_cpu, E_cpu = X.cpu(), E.cpu()
        for i in range(count):
            xi, ei = X_cpu[i], E_cpu[i]
            n_v, n_a = atom_valence_violations(xi, ei)
            total_violations += n_v
            total_atoms += n_a
            info = analyze_molecule(xi, ei)
            if info["strict_valid"]:
                n_strict += 1
                strict_mols.append(info["mol"])
            if info["frag_valid"]:
                n_frag += 1

validity_strict = n_strict / N_EVAL
validity_frag = n_frag / N_EVAL
atom_violation_rate = (total_violations / total_atoms) if total_atoms > 0 else float("nan")
smiles_set = {Chem.MolToSmiles(m) for m in strict_mols}
uniqueness = len(smiles_set) / len(strict_mols) if strict_mols else 0.0

print(f"Validity (strict): {validity_strict:.3f}  ({n_strict}/{N_EVAL})")
print(f"Validity (largest-fragment, 문헌 비교용): {validity_frag:.3f}  ({n_frag}/{N_EVAL})")
print(f"Atom violation rate: {atom_violation_rate:.3f}  ({total_violations}/{total_atoms} atoms)")
print(f"Uniqueness: {uniqueness:.3f}")

if strict_mols:
    sampled = random.sample(strict_mols, min(16, len(strict_mols)))
    print(f"\nSMILES (random {min(10, len(strict_mols))}):")
    for mol in random.sample(strict_mols, min(10, len(strict_mols))):
        print(" ", Chem.MolToSmiles(mol))
    img = Draw.MolsToGridImage(sampled, molsPerRow=4, subImgSize=(300, 300))
    display(img)
else:
    print("\n유효한(strict) 분자가 생성되지 않았습니다.")
